In [1]:
import gcsfs
import xarray as xr
import time
import os
import numpy as np
from pathlib import Path

## IFS ENS mean

In [2]:
fs = gcsfs.GCSFileSystem(token="anon")

ds_pangu = xr.open_zarr(
    "gs://weatherbench2/datasets/ifs_ens/2018-2022-1440x721_mean.zarr",
    consolidated=True,
    storage_options={"token": "anon"}
)

In [3]:
print(ds_pangu)

<xarray.Dataset> Size: 27TB
Dimensions:                   (time: 3652, prediction_timedelta: 61,
                               latitude: 721, longitude: 1440, level: 3)
Coordinates:
  * time                      (time) datetime64[ns] 29kB 2018-01-01 ... 2022-...
  * prediction_timedelta      (prediction_timedelta) int64 488B 0 6 ... 354 360
  * latitude                  (latitude) float32 3kB -90.0 -89.75 ... 89.75 90.0
  * longitude                 (longitude) float32 6kB 0.0 0.25 ... 359.5 359.8
  * level                     (level) int32 12B 500 700 850
Data variables: (12/15)
    10m_u_component_of_wind   (time, prediction_timedelta, latitude, longitude) float32 925GB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    10m_v_component_of_wind   (time, prediction_timedelta, latitude, longitude) float32 925GB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    10m_wind_speed            (time, prediction_timedelta, latitude, longitude) float32 925GB dask.array<chunk

In [4]:
# einzelne variabeln und zeiträume als nc speichern
start_time = time.time()
temperature = ds_pangu["mean_sea_level_pressure"].sel(
    time=slice("2020-12-01", "2021-02-28"),
    prediction_timedelta=slice(6, 96),
    latitude=slice(12, 84),
    longitude=((ds_pangu.longitude <= 50) | (ds_pangu.longitude >= 270))
)

# Longitude von -90° bis 50° umrechnen und sortieren
temperature = temperature.assign_coords(
    longitude=((temperature.longitude + 180) % 360) - 180
).sortby("longitude")

savepath = 'data/ifs_mean/mean_sea_level_pressure/ifs_mean_mean_sea_level_pressure_2020-2021_DJF.nc'
os.makedirs(os.path.dirname(savepath), exist_ok=True)

temperature.to_netcdf(savepath)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

Time taken: 1011.0894401073456 seconds


In [5]:
stop

NameError: name 'stop' is not defined

## Pangu ##

In [2]:
fs = gcsfs.GCSFileSystem(token="anon")

ds_pangu = xr.open_zarr(
    "gs://weatherbench2/datasets/pangu/2018-2022_0012_0p25.zarr",
    consolidated=True,
    storage_options={"token": "anon"}
)

In [3]:
# einzelne variabeln und zeiträume als nc speichern
start_time = time.time()
temperature = ds_pangu["v_component_of_wind"].sel(
    time=slice("2021-12-01", "2022-02-28"),
    prediction_timedelta=slice(6, 96),
    latitude=slice(84, 12),
    longitude=((ds_pangu.longitude <= 50) | (ds_pangu.longitude >= 270)),
    level=300
)

# Longitude von -90° bis 50° umrechnen und sortieren
temperature = temperature.assign_coords(
    longitude=((temperature.longitude + 180) % 360) - 180
).sortby("longitude")

savepath = 'data/pangu/v_component_of_wind/pangu_v_component_of_wind_2021-2022_DJF.nc'
os.makedirs(os.path.dirname(savepath), exist_ok=True)

temperature.to_netcdf(savepath)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

Time taken: 5755.240826129913 seconds


In [ ]:
stop

## RH aus T und specific humidity berechnen

In [ ]:
T = xr.open_dataset('data/pangu/temperature/pangu_temperature_2019-2020_DJF.nc')
SH = xr.open_dataset('data/pangu/specific_humidity/pangu_specific_humidity_2019-2020_DJF.nc')

In [ ]:
p = 700 #pascal
T0 = 273.15 #Kelvin

In [ ]:
# RH_vals = 0.263*p*SH.specific_humidity*(np.exp(17.67*(T.temperature-T0)/(T.temperature-29.65)))**(-1)

In [ ]:
e_sw = np.exp(54.842763 - 6763.22/T.temperature - 4.210*np.log(T.temperature) + 0.000367*T.temperature + np.tanh(0.0415*(T.temperature-218.8))*(53.878 - 1331.22/T.temperature - 9.44523*np.log(T.temperature) + 0.014025*T.temperature))

In [ ]:
RH_vals = SH.specific_humidity/e_sw * p * 100/0.622

In [ ]:
relative_humidity = RH_vals.to_dataset(name="relative_humidity")

In [ ]:
relative_humidity

<xarray.Dataset> Size: 2GB
Dimensions:               (latitude: 289, prediction_timedelta: 16, time: 180,
                           longitude: 561)
Coordinates:
  * latitude              (latitude) float32 1kB 84.0 83.75 83.5 ... 12.25 12.0
  * prediction_timedelta  (prediction_timedelta) int64 128B 6 12 18 ... 84 90 96
  * time                  (time) datetime64[ns] 1kB 2019-12-01 ... 2020-02-28...
  * longitude             (longitude) float32 2kB -90.0 -89.75 ... 49.75 50.0
    level                 int64 8B 700
Data variables:
    relative_humidity     (time, prediction_timedelta, latitude, longitude) float32 2GB ...

In [ ]:
savepath = 'data/pangu/relative_humidity/pangu_relative_humidity_2019-2020_DJF.nc'
os.makedirs(os.path.dirname(savepath), exist_ok=True)
relative_humidity.to_netcdf(savepath)

In [2]:
def specific_humidity_to_relative_humidity(q, T, p):
#     """
#     Converts specific humidity to relative humidity.
#     Args:
#         q (float): Specific humidity in kg/kg
#         T (float): Temperature in Kelvin
#         p (float): Pressure in hPa
#     """
    T_c = T - 273.15 # convert to Celsius
    e = (q * p) / (0.622 + 0.378 * q) # vapor pressure
    # saturation vapor pressure in hPa:
    es_water = 6.112 * np.exp((17.67 * T_c) / (T_c + 243.5))
    es_ice = 6.112 * np.exp((22.46 * T_c) / (T_c + 272.62))
    es = np.where(T_c >= 0, es_water, es_ice)

    RH = (e / es) * 100 # relative humidity in %
    return RH

In [24]:
model = 'pangu'
path = Path(f'data/{model}/')
ds_q = xr.open_dataset(path / f"specific_humidity/{model}_specific_humidity_2019-2020_DJF.nc")
ds_T = xr.open_dataset(path / f"temperature/{model}_temperature_2019-2020_DJF.nc")

In [34]:
# q = ds_q["specific_humidity"] 
q = ds_q.specific_humidity[:-2,:,:,:]
T = ds_T["temperature"]


# Druck
p = 7  # hPa


# --------------------------------------------------
# Relative Feuchte berechnen
# --------------------------------------------------

rh = specific_humidity_to_relative_humidity(q, T, p)


# Als Dataset speichern
ds_rh = rh.to_dataset(name="relative_humidity")

In [ ]:
savepath = Path(f"data/{model}/relative_humidity/")
os.makedirs(savepath, exist_ok=True)
ds_rh.to_netcdf(savepath / f"{model}_relative_humidity_2019-2020_DJF.nc")

In [9]:
ds_rh

<xarray.Dataset> Size: 2GB
Dimensions:               (latitude: 289, prediction_timedelta: 16, time: 180,
                           longitude: 561)
Coordinates:
  * latitude              (latitude) float32 1kB 12.0 12.25 12.5 ... 83.75 84.0
  * prediction_timedelta  (prediction_timedelta) int64 128B 6 12 18 ... 84 90 96
  * time                  (time) datetime64[ns] 1kB 2019-12-01 ... 2020-02-28...
  * longitude             (longitude) float32 2kB -90.0 -89.75 ... 49.75 50.0
    level                 int32 4B 700
Data variables:
    relative_humidity     (time, prediction_timedelta, latitude, longitude) float32 2GB ...

## Daten zusammenfügen

In [3]:
var = "v_component_of_wind"
model = 'pangu'
path = Path(f'data/{model}/{var}/')
dateien = sorted(path.glob("*.nc"))
print(dateien)
# Dateien entlang der Koordinate "time" zusammenfügen
ds = xr.open_mfdataset(
    dateien,
    combine="nested",
    parallel=False,
    compat="override"
)
ds = ds.sortby("time")
# Ergebnis speichern
ds.to_netcdf(path / f"{model}_{var}_2019-2022_DJF.nc")

print(ds)

[WindowsPath('data/pangu/v_component_of_wind/pangu_v_component_of_wind_2019-2020_DJF.nc'), WindowsPath('data/pangu/v_component_of_wind/pangu_v_component_of_wind_2020-2021_DJF.nc'), WindowsPath('data/pangu/v_component_of_wind/pangu_v_component_of_wind_2021-2022_DJF.nc')]


C:\Users\Luisa\AppData\Local\Temp\ipykernel_33944\15315124.py:7: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  ds = xr.open_mfdataset(


<xarray.Dataset> Size: 6GB
Dimensions:               (latitude: 289, prediction_timedelta: 16, time: 542,
                           longitude: 561)
Coordinates:
  * latitude              (latitude) float32 1kB 84.0 83.75 83.5 ... 12.25 12.0
  * prediction_timedelta  (prediction_timedelta) int64 128B 6 12 18 ... 84 90 96
  * time                  (time) datetime64[ns] 4kB 2019-12-01 ... 2022-02-28...
  * longitude             (longitude) float32 2kB -90.0 -89.75 ... 49.75 50.0
    level                 int64 8B ...
Data variables:
    v_component_of_wind   (time, prediction_timedelta, latitude, longitude) float32 6GB dask.array<chunksize=(182, 16, 289, 561), meta=np.ndarray>


In [2]:
import xarray as xr
from pathlib import Path

var = "v_component_of_wind"
model = "pangu"

path = Path(f"data/{model}/{var}/")
dateien = sorted(path.glob("*.nc"))

datasets = [xr.open_dataset(f) for f in dateien]

# Dateien entlang der Zeitdimension zusammenfügen
ds = xr.concat(
    datasets,
    dim="time",
    data_vars="all",
    coords="minimal",
    compat="override",
    join="exact"
)

ds = ds.sortby("time")

ds.to_netcdf(path / f"{model}_{var}_2019-2022_DJF.nc")

for d in datasets:
    d.close()

print(ds)

<xarray.Dataset> Size: 6GB
Dimensions:               (time: 542, prediction_timedelta: 16, latitude: 289,
                           longitude: 561)
Coordinates:
  * time                  (time) datetime64[ns] 4kB 2019-12-01 ... 2022-02-28...
  * prediction_timedelta  (prediction_timedelta) int64 128B 6 12 18 ... 84 90 96
  * latitude              (latitude) float32 1kB 84.0 83.75 83.5 ... 12.25 12.0
  * longitude             (longitude) float32 2kB -90.0 -89.75 ... 49.75 50.0
    level                 int64 8B ...
Data variables:
    v_component_of_wind   (time, prediction_timedelta, latitude, longitude) float32 6GB ...


In [6]:
ds = xr.open_dataset('data/pangu/v_component_of_wind/pangu_v_component_of_wind_2021-2022_DJF.nc')

In [3]:
ds.level.values

array(300)

In [4]:
ds.v_component_of_wind[-1].values

array([[[-2.83136196e+01, -2.83805752e+01, -2.84120960e+01, ...,
         -5.50991631e+00, -5.22978783e+00, -5.09374666e+00],
        [-2.81437492e+01, -2.82643127e+01, -2.83420601e+01, ...,
         -6.33742046e+00, -6.09519529e+00, -5.95080853e+00],
        [-2.78763103e+01, -2.80490246e+01, -2.81747780e+01, ...,
         -7.19375515e+00, -6.98983622e+00, -6.83897018e+00],
        ...,
        [-2.38010597e+00, -2.93881798e+00, -3.40590262e+00, ...,
         -1.09700465e+00, -1.12665403e+00, -1.22695744e+00],
        [-1.99998224e+00, -2.67035842e+00, -3.28422761e+00, ...,
         -1.45878375e+00, -1.47619510e+00, -1.41673839e+00],
        [-1.79342389e+00, -2.52974606e+00, -3.31454182e+00, ...,
         -1.73345876e+00, -1.66658771e+00, -1.64007032e+00]],

       [[-3.41802597e+01, -3.44612083e+01, -3.46774483e+01, ...,
         -4.47582150e+00, -4.24803782e+00, -3.87236357e+00],
        [-3.31996536e+01, -3.35203934e+01, -3.37977638e+01, ...,
         -5.43788815e+00, -5.20076036e